## Penman-Monteith Hourly ET Estimates using ERA5

For hourly reference ET<sub>o</sub> calculations, use the [FAO Penman-Monteith](http://www.fao.org/3/X0490E/x0490e06.htm) equation:

$$ \lambda ET_0 = \frac{\Delta (R_n - G) + \frac{\rho_a c_p (e_s - e_a)}{r_a}}
             {(\Delta + \gamma (1 + \frac{r_s}{r_a}))\rho_w}$$

The full form above can be simplified by assuming values for an alfalfa reference crop, with fixed surface resistance of 45 s/m, and an albedo value of 0.23 ([Allen et al., 1998](http://www.fao.org/3/x0490e/x0490e00.htm); [Allen et al., 2006](doi.org/10.1016/j.agwat.2005.03.007)). Thus, the FAO-PM version is:

$$ ET_0 = \frac{0.408 \Delta (R_n - G) + \gamma \frac{C_n}{T + 273} u_2 (e_s - e_a)}
             {\Delta + \gamma (1 + C_d u_2)}$$

where

1) $ET_0$ is the reference evapotranspiration (in mm/hr) for a hypothetical alfalfa reference surface

2) $\Delta$, the slope of the saturation vapor pressure curve, can be calculated from the hourly air temperature (in &deg;C)

3) $R_n$ is net radiation (MJ/m<sup>2</sup>/hr), calculated from net long-wave and short-wave radiation (in W/m<sup>2</sup>)

4) $G$ is the soil heat flux (MJ/m<sup>2</sup>/hr)

Assume that for daytime hours ($R_n \gt 0$), $G = 0.1 \times R_n$.
For nighttime hours ($R_n \le 0$), $G = 0.5 \times R_n$.

5) $\gamma$ (kPa/&deg;C) is the psychrometric constant, which can be estimated from barometric pressure, P in kPa:

$$ \gamma = 0.000665 P $$

6) $C_n$ is a constant (K mm s<sup>3</sup>/mg/hr), 66 for hourly estimates

7) $T$ (&deg;C), mean hourly temperature at 2 m height

8) $u_2$ is the mean wind speed at 2 m height (m/s), calculated from wind speed at 10 m height assuming FAO 56 logarithmic wind profile:

$$ u_2 = u_z \frac{4.87}{ln(67.8z - 5.42)}$$

where $u_z$ is the wind speed at height z (in m), and z is the height of the wind measurement (10 m for ERA5)

9) $e_s$ is the saturation vapor pressure (kPa). Saturation vapor pressure is calculated from temperature $T$ (`t2m`) using the Magnus-Tetens formula:

$$e_{(T)} = 0.6108 * exp\big(\frac{17.27T}{T+237.3}\big)$$

10) $e_a$ is the actual vapor pressure ($e_a$) is calculated from the dewpoint temperature `d2m` using the Magnus-Tetens formula:

$$e_a = e_{(T)} * \frac{RH}{100}$$

Note

11) $C_d$ is a constant, 0.25/1.7 for day/night values in hourly estimates. ("Night" is defined as anytime that $R_n \le 0$.)

Note: For unit conversions, note that 1 W/m<sup>2</sup> = 0.0036 MJ/m<sup>2</sup>/hr

Also note that $C_n$ and $C_d$ are different for grass and alfalfa reference crops:

| Parameter             | Alfalfa | Grass |
| --------------------- | ------- | ----- |
| Daily Cn              |  1600   | 900   |
| Hourly Cn             |  66     | 37    |
| Daily Cd              |  0.38   | 0.34  |
| Hourly Cd (daytime)   | 0.25    | 0.24  |
| Hourly Cd (nighttime) | 1.7     | 0.96  |

In [ ]:
import s3fs
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def calc_delta(temperature):
    """Calculate the slope of the saturation vapor pressure curve according to
    the FAO definition.

    Parameters
    ----------
    temperature : float
        Temperature in degrees Celsius.

    Returns
    -------
    delta : float
        Slope of the saturation vapor pressure curve."""

    num = 4098*(0.6108 * np.exp((17.27*temperature)/(temperature+237.3)))
    denom = (temperature + 237.3)**2

    delta = num/denom

    return delta


def magnes_tetens(temperature):
    """Calculate the saturation vapor pressure for a given temperature

    Parameters
    ----------
    temperature : float
        Temperature in degrees Celsius.

    Returns
    -------
    pressure : float
        Vapor pressure at given temperature."""

    pressure = 0.6108*np.exp((17.27 * temperature)/(temperature + 237.3))

    return pressure

In [ ]:
# Load in ERA5 land data
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'
zarr_path = f'{s3_base_path}/input-data/raw-data/era5-land/era5-land_2001-01-01_2020-12-31_hourly.zarr'
fs = s3fs.S3FileSystem(anon=False)
ds = xr.open_zarr(zarr_path, storage_options={"anon": False})

print(ds)

# Within ds['site_id'], replace "Houston_Black" with "HoustonBlack"
if 'Houston_Black' in np.unique(ds['site_id']):
    hb_mask = ds['site_id'] != 'Houston_Black'
    ds = ds.assign_coords(site_id=ds['site_id'].where(hb_mask, 'HoustonBlack'))

# Load in csv of site locations/names
site_df = pd.read_csv(f'{s3_base_path}/sites/site-locations.csv')

In [ ]:
# For each site, check number of NaN values in each variable
pm_vars = ['d2m', 'sp', 'ssr', 'str', 't2m', 'tp', 'u10', 'v10']
nan_counts = (
    ds[pm_vars]
    .isnull()
    .sum("valid_time")
    .to_dataframe()
    .reset_index()
    .set_index("site_id")[pm_vars]
)

print(nan_counts)

The first value for accumulated variables (`ssr`, `str`, and `tp`) is NaN, which makes sense. However, there are two extra NaN values for `str` for Palouse.

In [ ]:
# For accumulated variables, simply copy the second time step to the first time step
for var in ['tp', 'ssr', 'str']:
    ds[var][{"valid_time": 0}] = ds[var].isel(valid_time=1)

# Interpolate over NaN values for `str` for Pullman
da = ds['str'].sel(site_id='Palouse').chunk({'valid_time': -1})
nan_times = da.valid_time[da.isnull().values]

for t in nan_times.values[-1:]:
    i = da.get_index("valid_time").get_loc(t)
    window = da.isel(valid_time=slice(max(i - 5, 0), i + 6))

    print(f"\nNaN values in `str` for Palouse")
    print(window.to_series())

# Linearly interpolate over NaNs
da_interp = da.interpolate_na(dim='valid_time', method='linear')

for t in nan_times.values[-1:]:
    i = da.get_index("valid_time").get_loc(t)
    window = da_interp.isel(valid_time=slice(max(i - 5, 0), i + 6))

    print(f"\nFilled NaN values in `str` for Palouse")
    print(window.to_series())

# Put it back into ds for that site only
ds['str'].loc[{'site_id': 'Palouse'}] = da_interp

In [ ]:
# Re-check number of NaN values in each variable
pm_vars = ['d2m', 'sp', 'ssr', 'str', 't2m', 'tp', 'u10', 'v10']
nan_counts = (
    ds[pm_vars]
    .isnull()
    .sum("valid_time")
    .to_dataframe()
    .reset_index()
    .set_index("site_id")[pm_vars]
)

print(nan_counts)

In [ ]:
## For hourly ET, calculate individual pieces
# 2. delta, the slope of the saturation vapor pressure curve
delta = calc_delta(ds['t2m'] - 273.15)

# 3. Rn, net radiation (in MJ/m2/hr)
# Note that we convert from W/m2 to MJ/m2/hr by multiplying by 0.0036
Rn = (ds['ssr'] + ds['str']) * 0.0036

# 4. G, soil heat flux
daylight_mask = Rn > 0
G = xr.where(daylight_mask, 0.1 * Rn, 0.5 * Rn)

# 5. gamma, the psychrometric constant (kPa/deg C)
# Note that we divide 'sp' by 1000 to convert from Pa to kPa)
gamma = 0.000665 * (ds['sp']/1000)

# 6. Cn, a constant
Cn = 66  # For alfalfa
Cn = 37  # For grass

# 7. Tmean, the mean hourly temperature in deg C (-273.15 to convert from K to C)
tmean = ds['t2m'] - 273.15

# 8. u, mean wind speed
# Calculate wind speed at 10 m from x and y components
s10 = np.sqrt(ds['u10']**2 + ds['v10']**2)
# Convert to wind speed at 2 m height using FAO56 assumption of log speed profile
u = s10 * 4.87/np.log(67.8*10 - 5.42)

# 9. es, the saturation vapor pressure
es = magnes_tetens(tmean)

# 10. ea, the actual vapor pressure
ea = magnes_tetens(ds['d2m'] - 273.15)

# 11. Cd, a constant
Cd = xr.where(daylight_mask, 0.25, 1.7)  # For alfalfa
Cd = xr.where(daylight_mask, 0.24, 0.96)  # For grass

## Combine individual pieces into hourly ET calc
num_left = 0.408*delta*(Rn - G)
num_right = gamma * Cn / (tmean + 273) * u * (es - ea)
num = num_left + num_right
denom = delta + gamma*(1 + Cd*u)

hourly_et = num/denom

# Ensure reference ET is always positive
hourly_et = hourly_et.clip(min=0)

# Convert to dataframe and label columns
et_df = hourly_et.to_pandas()
et_df.columns = site_df['site_id']

# Trim to 2001-2020 for a total of 20 years
et_df = et_df.loc['2001-01-01':'2020-12-31']

et_df.to_csv(f'{s3_base_path}/input-data/processed-data/climate/ref_et.csv')

In [ ]:
# Extract precipitation data for the same time period to compare with ET
ppt_df = ds['tp'].clip(min=0).to_pandas()
ppt_df /= 24  # Convert from mm/d to mm/hr
ppt_df = ppt_df.loc['2001-01-01':'2020-12-31']
ppt_df.columns = site_df['site_id']

ppt_df.to_csv(f'{s3_base_path}/input-data/processed-data/climate/precip.csv')

In [ ]:
# Also save net radiation
rn_df = Rn.to_pandas()
rn_df.columns = site_df['site_id']
rn_df.to_csv(f'{s3_base_path}/input-data/processed-data/climate/rn.csv')

In [ ]:
# Basic summary
print("Precipitation summary:")
print(ppt_df.describe().T[["min", "mean", "max"]])

print("\nET summary:")
print(et_df.describe().T[["min", "mean", "max"]])

In [ ]:
# Plot reference ET for each site
fig, ax = plt.subplots(et_df.shape[1], figsize=(6, 10), tight_layout=True, sharex=True)

for i, site in enumerate(site_df['site_id']):

    ax[i].plot(et_df.index, et_df[site], color='green')
    ax[i].set(ylabel='$ET_0$ (mm/hr)', title=site)

In [ ]:
# Plot precipitation for each site
fig, ax = plt.subplots(et_df.shape[1], figsize=(6, 10), tight_layout=True, sharex=True)

for i, site in enumerate(site_df['site_id']):

    ax[i].plot(ppt_df.index, ppt_df[site], color='steelblue')
    ax[i].set(ylabel='P (mm/hr)', title=site)